###Configuración Inicial y Esquema

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from delta.tables import DeltaTable

# 1. creación del esquema Bronze en el catálogo
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

# 2. Definir rutas base 
landing_base_path = "/Volumes/workspace/default/landing/"

print(" Entorno Bronze configurado.")

###Lógica de Ingesta

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from delta.tables import DeltaTable

# Definir esquema explícito
sales_schema = StructType([
    StructField("fecha_venta", StringType(), True),
    StructField("pais", StringType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("producto", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("valor_unitario", DoubleType(), True)
])

bronze_ventas_table = "workspace.bronze.ventas"
landing_base_path = "/Volumes/workspace/default/landing/"

# Leer CSV crudo
df_raw_sales = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(sales_schema)
    .load(f"{landing_base_path}ventas_csv/")
)


df_raw_sales.write.format("delta").mode("overwrite").saveAsTable(bronze_ventas_table)

print(f"Ingesta completada Ventas (CSV) -> {bronze_ventas_table}")

### Ingesta Cruda de CLIENTES

In [0]:
bronze_clientes_table = "workspace.bronze.clientes"

# Leer Parquet crudo
df_raw_clientes = spark.read.parquet(f"{landing_base_path}clientes_sql/")

df_raw_clientes.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.clientes")

print(f"Ingesta completada: Clientes ) -> {bronze_clientes_table}")

###Ingesta Cruda de INVENTARIO

In [0]:
bronze_inventario_table = "workspace.bronze.inventario"

# Leer JSON
df_raw_inventario = spark.read.json(f"{landing_base_path}inventario_api/")

df_raw_inventario.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.inventario")

print(f"ngesta completada: Inventario (JSON) -> {bronze_inventario_table}")